## Reading in all subject-level data, combining them into larger structures, saving results for statistical analysis, and performing basic plots


### 0. Set up everyting

In [1]:
import pandas as pd
import glob
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings("ignore")
import scipy.stats as stats

def jitter(values,j):
    return values + np.random.normal(j,0.1,values.shape)

folders = glob.glob("../data/00_gaze_data/sub*")
folders.sort()
subjs = []
for f in folders:
    subjs.append(f[-6:])
print(len(subjs))   
print(subjs)

# create a list of condition values (which is stored as 1/2 in the main subject memory file
memory_df = pd.read_csv('../data/01_interview_data/Recall_sheet_vr_driving_city.csv')

condition_index = {}
for i, subj in enumerate(subjs):
    condition_index[subj] = memory_df.loc[i,'condition']

print(condition_index)
memory_df.head()

43
['sub001', 'sub002', 'sub003', 'sub004', 'sub005', 'sub006', 'sub007', 'sub008', 'sub009', 'sub010', 'sub011', 'sub012', 'sub013', 'sub014', 'sub015', 'sub016', 'sub017', 'sub018', 'sub019', 'sub020', 'sub021', 'sub022', 'sub023', 'sub024', 'sub025', 'sub026', 'sub027', 'sub028', 'sub029', 'sub030', 'sub031', 'sub032', 'sub033', 'sub034', 'sub035', 'sub036', 'sub037', 'sub038', 'sub039', 'sub040', 'sub041', 'sub042', 'sub043']
{'sub001': 'Free_Viewing', 'sub002': 'Count_Sales_Signs', 'sub003': 'Free_Viewing', 'sub004': 'Count_Sales_Signs', 'sub005': 'Free_Viewing', 'sub006': 'Count_Sales_Signs', 'sub007': 'Free_Viewing', 'sub008': 'Count_Sales_Signs', 'sub009': 'Free_Viewing', 'sub010': 'Count_Sales_Signs', 'sub011': 'Free_Viewing', 'sub012': 'Count_Sales_Signs', 'sub013': 'Free_Viewing', 'sub014': 'Count_Sales_Signs', 'sub015': 'Free_Viewing', 'sub016': 'Free_Viewing', 'sub017': 'Count_Sales_Signs', 'sub018': 'Count_Sales_Signs', 'sub019': 'Free_Viewing', 'sub020': 'Free_Viewing', 

,subject,ad_brunch,ad_burger,ad_coffee,ad_pizza,ad_furniture,ad_hotel,ad_lawyer,ad_smartphone,ad_fundraising,...,health_jogging,health_getsleep,health_doctor,health_caffeine,health_greatlake,health_mentalhealth,health_recycling,health_polarbears,condition,Note
0,sub001,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Free_Viewing,NaN
1,sub002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Count_Sales_Signs,NaN
2,sub003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,Free_Viewing,NaN
3,sub004,NaN,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Count_Sales_Signs,NaN
4,sub005,NaN,1.0,NaN,NaN,NaN,1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Free_Viewing,NaN


### 1. Create the LargeDf 

This df stacks all information, i.e. every subject (of 40), every billboard (of 20), whether it was fixated, how often, for how long overall (if fixated), and whether it was recalled and recognized.
This also forms the basis for the main statistical analyses

In [2]:
list_of_dataframes = []
for i in range(len(subjs)): 
    read_file_name = '../data/03_fixation_billboards_recall_recognition/'  + subjs[i] +'_fixations_billboards_recall_recognition_integrated.csv'
    df = pd.read_csv(read_file_name, index_col = 0)
    df['subject'] = subjs[i]
    df["condition"] = condition_index[subjs[i]]

    #make all a bit prettier
    df['billboard#_fixated'] = np.where(df['fix_bill'] > 0, 1, 0)
    new_column_names = {'position_index': 'position_index', 'billboard_image_at_position_text': 'billboard_image_at_position', 'billboard_image_at_position_text_attention': 'billboard_full_name',
                       'fix_bill': 'fix_bill', 'billboard#_fixated_count': 'billboard#_fixated_count', 'billboard#_duration': 'billboard#_duration', 
                       'recall': 'recall', 'recognition': 'recognition', 'subject': 'subject',   'condition': 'condition'}
    df = df.rename(columns=new_column_names)
    df['billboard_attention_level'] = df['billboard_full_name'].apply(lambda x: 'more_attention' if 'moreatt_' in x else 'less_attention')

    df = df.drop(columns=['fix_bill', 'billboard_full_name'])
    
    # Replace the column names and order as needed
    new_column_order = ['subject', 'condition', 'position_index', 'billboard_image_at_position', 'billboard_attention_level', 
                       'billboard#_fixated', 'billboard#_fixated_count', 'billboard#_duration', 'recall', 'recognition']
    df = df[new_column_order]
    
    list_of_dataframes.append(df)
    
large_df = pd.concat(list_of_dataframes)

#commenting out saving
large_df.to_csv('../data/04_data_analysis/00_all_data.csv')

large_df.tail()

,subject,condition,position_index,billboard_image_at_position,billboard_attention_level,billboard#_fixated,billboard#_fixated_count,billboard#_duration,recall,recognition
35,sub043,Count_Sales_Signs,36,health_smoking,less_attention,1,1.0,0.58,0,0.0
36,sub043,Count_Sales_Signs,37,ad_fundraising,more_attention,1,3.0,1.24,0,1.0
37,sub043,Count_Sales_Signs,38,health_employee,less_attention,1,2.0,1.04,0,0.0
38,sub043,Count_Sales_Signs,39,ad_carpetcleaning,more_attention,1,1.0,2.22,1,1.0
39,sub043,Count_Sales_Signs,40,ad_hotel,less_attention,1,1.0,0.45,0,0.0


### 2. Create the BySubjectDf

This df pivots the data by subjects, i.e. so that we have subject level information about condition, number of unique billboards fixated, duration etc.... We also compute rates for the fixation, recall, and recognition, respectively.
This forms the basis for the initial analyses: Effects of Driving Condition and BB Emotionality on Fixations, 
Then on Memory....

In [ ]:
# bysubject_df = large_df.groupby(['subject', 'condition', 'billboard_attention_level']).agg({'billboard#_fixated':'sum', 'billboard#_duration':'sum', 'recall':'sum', 'recognition':'sum'})
# bysubject_df.reset_index(inplace=True)

# bysubject_df['fixated_rate']      = bysubject_df['billboard#_fixated']/20
# bysubject_df['recall_rate']      = bysubject_df['recall']/20
# bysubject_df['recognition_rate'] = bysubject_df['recognition']/20
# bysubject_df = bysubject_df.drop(columns=['recall', 'recognition', 'billboard#_fixated'])

# bysubject_df.head()

# g2 = bysubject_df.pivot(index='subject', columns='billboard_attention_level', values = ['condition', 'fixated_rate', 'billboard#_duration', 'recall_rate', 'recognition_rate'] )
# g2.columns = g2.columns.map('_'.join)
# g2 = g2.drop(columns=['condition_more_attention'])
# g2.rename(columns={'condition_less_attention': 'condition'}, inplace=True)
# g2.to_csv('../data/04_data_analysis/01_bysubject_aggregate_data.csv')
# g2.head()

### Gradient Analysis

The goal of this analysis is to go over every individual participant, find out how often they fixated a given billboard (same could be done with the duration). We will then form three bins (taking into account variability across subjects): Billboards that were never fixated, billboards that the given subject fixated somewhat, but not very often; and finally, billboards that the subject fixated multiple times.

In [ ]:
# # for recall
# subjres = []
# for i in range(len(subjs)):
#     res = np.zeros(3)
    
#     df = large_df[large_df['subject'] == subjs[i]]
#     curr_sub_med_fix = df[ df['billboard#_fixated_count']!=0]['billboard#_fixated_count'].median()

#     res[0] = df[ df['billboard#_fixated_count']==0]['recall'].sum()

#     interim_df = df[ df['billboard#_fixated_count'] > 0 ]
#     interim_df = interim_df[ interim_df['billboard#_fixated_count'] < curr_sub_med_fix ] #less than median fixation
#     res[1] = interim_df['recall'].sum()    

#     res[2] = df[ df['billboard#_fixated_count']>=curr_sub_med_fix]['recall'].sum()  # more than median fixations (
#     subjres.append(res)
    
# dosedf_recall = pd.DataFrame(np.asarray(subjres), columns = ['not_fixated_recalled', 'more_fixated_recalled','very_fixated_recalled'])
# dosedf_recall['subject'] = subjs

# # for recognition
# subjres = []

# for i in range(len(subjs)):
#     res = np.zeros(3)
    
#     df = large_df[large_df['subject'] == subjs[i]]
#     curr_sub_med_fix = df[ df['billboard#_fixated_count']!=0]['billboard#_fixated_count'].median()

#     res[0] = df[ df['billboard#_fixated_count']==0]['recognition'].sum()

#     interim_df = df[ df['billboard#_fixated_count'] > 0 ]
#     interim_df = interim_df[ interim_df['billboard#_fixated_count'] < curr_sub_med_fix ] #less than median fixation
#     res[1] = interim_df['recognition'].sum()    

#     res[2] = df[ df['billboard#_fixated_count']>=curr_sub_med_fix]['recognition'].sum()  # more than median fixations (
#     subjres.append(res)
    
# dosedf_recognition = pd.DataFrame(np.asarray(subjres), columns = ['not_fixated_recognized', 'more_fixated_recognized','very_fixated_recognized'])
# dosedf_recognition['subject'] = subjs

# dosedf_recognition.head()

# dosedf_recall_recognition = dosedf_recall.merge(dosedf_recognition,  left_on = 'subject', right_on = 'subject')
# dosedf_recall_recognition['condition'] = dosedf_recall_recognition['subject'].map(condition_index)

# dosedf_recall_recognition = dosedf_recall_recognition.set_index('subject')
# new_column_order = ['condition', 'not_fixated_recalled', 'more_fixated_recalled', 'very_fixated_recalled', 
#                     'not_fixated_recognized', 'more_fixated_recognized', 'very_fixated_recognized']
# dosedf_recall_recognition = dosedf_recall_recognition[new_column_order]
# dosedf_recall_recognition.head()
# dosedf_recall_recognition.to_csv('../data/04_data_analysis/02_dosedf_fixation_recall_recognition.csv')

### Visualize Viewing Behavior as a Function of Driving Condition

In [ ]:
# fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(12, 4))

# by_pos_df_dis = large_df[large_df['condition']=='Count_Sales_Signs'].groupby('position_index').agg({'billboard#_fixated':'mean'})
# by_pos_df_no_dis = large_df[large_df['condition']=='Free_Viewing'].groupby('position_index').agg({'billboard#_fixated':'mean'})

# by_pos_df_no_dis.plot(label='Free_Viewing',color = 'black',  ax = ax1)
# by_pos_df_dis.plot(label='Count_Sales_Signs', color = 'black', linestyle='--', ax = ax1)

# ax1.set_ylim([0,1.05])
# ax1.set_ylabel('Probability of Fixation')
# ax1.set_xlabel('Billboard Position')
# ax1.set_xlim([1,20])
# ax1.set_xticks( [1, 5, 10, 15, 20])
# ax1.set_xticklabels( [1, 5, 10, 15, 20])
# ax1.legend([ "Free Viewing", "Sales Sign Counting"], loc='lower right');
# ax1.set_title('By Position: Fixation Probability by Driving Condition');

# # 
# by_pos_df_dis = large_df[large_df['condition']=='Count_Sales_Signs'].groupby('position_index').agg({'billboard#_duration':'mean'})
# by_pos_df_no_dis = large_df[large_df['condition']=='Free_Viewing'].groupby('position_index').agg({'billboard#_duration':'mean'})

# by_pos_df_no_dis.plot(label='Free_Viewing', color = 'black',  ax = ax2)
# by_pos_df_dis.plot(label='Count_Sales_Signs', color = 'black', linestyle='--',  ax = ax2)

# ax2.set_ylim([0,4.05])
# ax2.set_ylabel('Average Duration [s]')
# ax2.set_xlabel('Billboard Position')
# ax2.set_xlim([1,20])
# ax2.set_xticks( [1, 5, 10, 15, 20])
# ax2.set_xticklabels( [1, 5, 10, 15, 20])
# ax2.set_yticks(range(1, 5, 1));
# ax2.legend([ "Free Viewing", "Sales Sign Counting"], loc='lower right');
# ax2.set_title('By Position: Gaze Duration by Driving Condition');


we can see above that there's a strong effect of Driving Condition: Much more fixations in the free viewing condition (left) and much longer gaze duration (right)




### Visualize Viewing Behavior as a Function of Driving Condition

In [ ]:
# fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(12, 4))

# by_pos_df_more_attention = large_df[large_df['billboard_attention_level']=='more_attention'].groupby('position_index').agg({'billboard#_fixated':'mean'})
# by_pos_df_less_attention = large_df[large_df['billboard_attention_level']=='less_attention'].groupby('position_index').agg({'billboard#_fixated':'mean'})

# by_pos_df_more_attention.plot(label='More Attention Grabbing Billboards', ax = ax1, color = 'red')
# by_pos_df_less_attention.plot(label='Less Attention Grabbing Billboards', ax = ax1, color = 'blue')

# ax1.set_ylim([0,1.05])
# ax1.set_ylabel('Probability of Fixation')
# ax1.set_xlim([1,20])
# ax1.set_xticks( [1, 5, 10, 15, 20])
# ax1.set_xticklabels( [1, 5, 10, 15, 20])
# ax1.legend([ "More Attention Grabbing Billboards", "Less Attention Grabbing Billboards"], loc='lower right');
# ax1.set_title('Fixation Probability by Driving Condition');


# #
# by_pos_df_more_attention = large_df[large_df['billboard_attention_level']=='more_attention'].groupby('position_index').agg({'billboard#_duration':'mean'})
# by_pos_df_less_attention = large_df[large_df['billboard_attention_level']=='less_attention'].groupby('position_index').agg({'billboard#_duration':'mean'})

# by_pos_df_more_attention.plot(label='More Attention Grabbing Billboards', ax = ax2, color = 'red')
# by_pos_df_less_attention.plot(label='Less Attention Grabbing Billboards', ax = ax2, color = 'blue')

# ax2.set_ylim([0,4.05])
# ax2.set_ylabel('Average Duration [s]')
# ax2.set_xlim([1,20])
# ax2.set_xticks( [1, 5, 10, 15, 20])
# ax2.set_xticklabels( [1, 5, 10, 15, 20])
# ax2.set_yticks( [1,2,3,4])
# ax2.set_yticklabels( [1,2,3,4])
# ax2.legend([ "More Attention Grabbing Billboards", "Less Attention Grabbing Billboards"], loc='lower right');
# ax2.set_title('Fixation Probability by Driving Condition');




In [ ]:
res = large_df.groupby(['subject', 'condition']).agg({'billboard#_fixated':'sum', 'billboard#_duration':'sum', 'recall':'sum', 'recognition':'sum'})
res.reset_index(inplace=True)
res['recall_rate'] = res['recall']/40
res['recognition_rate'] = res['recognition']/40
#res.to_csv('../data/04_statistics/csv/billboard_fixations__durations_recall_recognition_bysubject_bycondition.csv')

In [ ]:
grouped = large_df.groupby(['subject', 'billboard_attention_level', 'condition']).agg({'billboard#_fixated':'sum', 'billboard#_duration':'sum',  'recall':'sum', 'recognition':'sum'})
grouped.reset_index(inplace=True)
grouped

more_less_att_fixbinarized = grouped.pivot(index='subject', columns='billboard_attention_level', values = ['billboard#_fixated', 'billboard#_duration', 'recall', 'recognition'] )
more_less_att_fixbinarized.columns = more_less_att_fixbinarized.columns.map('_'.join)
more_less_att_fixbinarized.to_csv('../data/04_data_analysis/03_billboard_fixation_duration_recall_recognition.csv')
more_less_att_fixbinarized.head()

In [ ]:
by_subject_df = large_df.groupby('subject').agg({'billboard#_fixated':'sum', 'recall':'sum', 'recognition':'sum'})
#by_subject_df = by_subject_df.drop(columns = "fix_binarized")
by_subject_df['recall_rate'] = by_subject_df['recall']/40 
by_subject_df['recognition_rate'] = by_subject_df['recognition']/40 
by_subject_df.tail()

### Plot

In [ ]:
large_df[['billboard#_fixated_count', 'billboard_attention_level']].groupby('billboard_attention_level').agg({'billboard#_fixated_count': 'mean'})

In [ ]:
plt.figure(figsize = (3,3,))
plt.scatter(x = jitter(large_df['billboard#_fixated'],0), 
                y = jitter(large_df.recall,0));

In [ ]:
# list with ordered categories
fig, ax = plt.subplots(1, figsize=(3,3))

sns.scatterplot(x = jitter(large_df['billboard#_fixated'],0), 
                y = jitter(large_df.recall,0),
                hue = large_df.condition,
                alpha=0.35,
                );
plt.legend(loc='right');

plt.xticks([0, 1], ['not looked at', 'looked at'])
plt.xlabel('BILLBOARD FIXATIONS');

plt.yticks([0, 1], ['not recalled', 'recalled'])
plt.yticks(rotation=90)
plt.ylabel('FREE RECALL');

sns.despine()

In [ ]:
# list with ordered categories
fig, ax = plt.subplots(1, figsize=(3,3))

sns.scatterplot(x = jitter(large_df['billboard#_fixated'],0), 
                y = jitter(large_df.recognition,0),
                hue = large_df.condition,
                alpha=0.35,
                );
plt.legend(loc='right');

plt.xticks([0, 1], ['not looked at', 'looked at'])
plt.xlabel('BILLBOARD FIXATIONS');

plt.yticks([0, 1], ['not recognized', 'recognized'])
plt.yticks(rotation=90)
plt.ylabel('RECOGNITION');

sns.despine()

In [ ]:
custom_palette = {'high_emo': 'red', 'low_emo': 'blue'}

fig, ax = plt.subplots(1, figsize=(3,3))

sns.scatterplot(x = jitter(large_df['billboard#_fixated'],0), 
                y = jitter(large_df.recall,0),
                hue = large_df.billboard_emotionality,
                alpha=0.35,
                palette=custom_palette, 
                );
plt.legend(loc='right');

plt.xticks([0, 1], ['not looked at', 'looked at'])
plt.xlabel('BILLBOARD FIXATIONS');

plt.yticks([0, 1], ['not recalled', 'recalled'])
plt.yticks(rotation=90)
plt.ylabel('FREE RECALL');

sns.despine()

In [ ]:
custom_palette = {'high_emo': 'red', 'low_emo': 'blue'}

fig, ax = plt.subplots(1, figsize=(3,3))

sns.scatterplot(x = jitter(large_df['billboard#_fixated'],0), 
                y = jitter(large_df.recognition,0),
                hue = large_df.billboard_emotionality,
                alpha=0.35,
                palette=custom_palette, 
                );
plt.legend(loc='right');

plt.xticks([0, 1], ['not looked at', 'looked at'])
plt.xlabel('BILLBOARD FIXATIONS');

plt.yticks([0, 1], ['not recognized', 'recognized'])
plt.yticks(rotation=90)
plt.ylabel('RECOGNITION');

sns.despine()

In [ ]:

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8,4))

ax1.set_title('Free Viewing')
sub_df = large_df[large_df['condition'] == 'free viewing']
custom_palette1 = {'high_emo': 'red', 'low_emo': 'blue'}
sns.scatterplot(data = sub_df,
                x = jitter(sub_df['billboard#_fixated'],0), 
                y = jitter(sub_df.recall,0),
                hue = sub_df.billboard_emotionality,
                alpha=0.35,
                palette=custom_palette1, 
                ax = ax1);

ax1.legend(loc='right');
ax1.set_xticks([0, 1], ['not looked at', 'looked at'])
ax1.set_xlabel('BILLBOARD FIXATIONS');
ax1.set_yticks([0, 1], ['not recalled', 'recalled'], rotation=90, verticalalignment='center')
ax1.set_ylabel('BILLBOARD RECALL');


ax2.set_title('Count Trash')
sub_df = large_df[large_df['condition'] == 'count trash']
custom_palette2 = {'high_emo': 'red', 'low_emo': 'blue'}
sns.scatterplot(data = sub_df,
                x = jitter(sub_df['billboard#_fixated'],0), 
                y = jitter(sub_df.recall,0),
                hue = sub_df.billboard_emotionality,
                alpha=0.35,
                palette=custom_palette2, 
                ax = ax2);

ax2.legend(loc='right');
ax2.set_xticks([0, 1], ['not looked at', 'looked at'])
ax2.set_xlabel('BILLBOARD FIXATIONS');
ax2.set_yticks([0, 1], [' ', ' '], rotation=90, verticalalignment='center')
ax2.set_ylabel(' ');

sns.despine()

In [ ]:

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8,4))

ax1.set_title('Free Viewing')
sub_df = large_df[large_df['condition'] == 'free viewing']
custom_palette1 = {'high_emo': 'red', 'low_emo': 'blue'}
sns.scatterplot(data = sub_df,
                x = jitter(sub_df['billboard#_fixated'],0), 
                y = jitter(sub_df.recognition,0),
                hue = sub_df.billboard_emotionality,
                alpha=0.35,
                palette=custom_palette1, 
                ax = ax1);

ax1.legend(loc='right');
ax1.set_xticks([0, 1], ['not looked at', 'looked at'])
ax1.set_xlabel('BILLBOARD FIXATIONS');
ax1.set_yticks([0, 1], ['not recognized', 'recognized'], rotation=90, verticalalignment='center')
ax1.set_ylabel('RECOGNITION');


ax2.set_title('Count Trash')
sub_df = large_df[large_df['condition'] == 'count trash']
custom_palette2 = {'high_emo': 'red', 'low_emo': 'blue'}
sns.scatterplot(data = sub_df,
                x = jitter(sub_df['billboard#_fixated'],0), 
                y = jitter(sub_df.recognition,0),
                hue = sub_df.billboard_emotionality,
                alpha=0.35,
                palette=custom_palette2, 
                ax = ax2);

ax2.legend(loc='right');
ax2.set_xticks([0, 1], ['not looked at', 'looked at'])
ax2.set_xlabel('BILLBOARD FIXATIONS');
ax2.set_yticks([0, 1], [' ', ' '], rotation=90, verticalalignment='center')
ax2.set_ylabel(' ');

sns.despine()

### Gradient analysis

In [ ]:
df

### large df

In [ ]:
large_df.head()

In [ ]:
grouped = large_df.groupby(['billboard_image_at_position_text', 'billboard_emotionality']).agg({'fix_binarized':'sum', 'fix_binarized':'sum', 'recall':'sum', 'recognition':'sum'})
grouped.reset_index(inplace=True)
grouped

g2 = grouped.pivot(index='billboard_image_at_position_text', columns='billboard_emotionality', values = ['fix_binarized', 'recall', 'recognition'] )
g2.columns = g2.columns.map('_'.join)
g2.to_csv('../data/04_statistics/csv/g2.csv')
g2


In [ ]:
sns.regplot(x = g2['recall_low_emo'],
            y = g2['recall_high_emo'])

sns.regplot(x = g2['recognition_low_emo'],
            y = g2['recognition_high_emo'])

plt.axis('square')
plt.xlim([-1,21])
plt.xticks(range(0, 21,5));
plt.xlabel('Recall/Recognition: Low Emo Version');

plt.ylim([-1,21])
plt.yticks(range(0, 21,5));
plt.ylabel('Recall/Recognition: High Emo Version');



#sns.despine()

print(np.corrcoef(g2['recall_high_emo'], g2['recognition_high_emo'])[0,1])
print(np.corrcoef(g2['recall_low_emo'], g2['recognition_low_emo'])[0,1])

print(np.corrcoef(g2['recall_high_emo'], g2['recall_low_emo'])[0,1])
print(np.corrcoef(g2['recognition_high_emo'], g2['recognition_low_emo'])[0,1])

In [ ]:
sns.regplot(x = g2['recall_low_emo'],
            y = g2['recognition_low_emo'])

sns.regplot(x = g2['recall_high_emo'],
            y = g2['recognition_high_emo'])

plt.axis('square')
plt.xlim([-1,21])
plt.xticks(range(0, 21,5));
plt.xlabel('Recall');

plt.ylim([-1,21])
plt.yticks(range(0, 21,5));
plt.ylabel('Recognition');


In [ ]:
large_df.to_csv('../data/04_statistics/csv/large_df.csv')

In [ ]:
g2.corr().round(decimals=2)

In [ ]:
plt.imshow(g2.corr(), cmap = 'seismic', vmin = -1, vmax = 1);
#plt.xtick_label(g2.columns.values)
plt.yticks(range(0,6), g2.columns.values)
plt.xticks(range(0,6), g2.columns.values, rotation = 90)

plt.colorbar();